# Forward-Pass Memory Cost: ALiBi-DINOv2 vs. DINOv2 (with SDPA)

This notebook measures the peak GPU memory overhead of an `eval`-mode forward pass (`torch.inference_mode()`, `float32`, batch size $B=1$) for **ALiBi-DINOv2** (`alibi_coco_dinov2_s`) compared to **DINOv2** (`dinov2_s`, ViT-S/14 with $H=6$ attention heads and $L=12$ blocks), with PyTorch Scaled Dot-Product Attention (`fused_attn = True`, `F.scaled_dot_product_attention`) enabled on both models, as a function of square input image size $S \times S$ ($S \le 1500$).

We attribute the memory difference to the theoretical memory cost of an $N_{\text{tokens}} \times N_{\text{tokens}}$ adjacency/distance matrix:

$$N_{\text{tokens}} = \left(\frac{S}{14}\right)^2 + 1_{\text{CLS}} + 4_{\text{REG}} = \left(\frac{S}{14}\right)^2 + 5, \qquad M_{\text{adj}}(N_{\text{tokens}}) = 4\, N_{\text{tokens}}^2 \text{ bytes (fp32)}.$$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from dinosaw.utils import add_custom_font
from dinosaw.wrappers import ModelTypes, get_models

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda:0"

In [ ]:
selected_models: tuple[ModelTypes, ...] = ("dinov2_s", "alibi_coco_dinov2_s")
models = get_models(
    selected_models,
    DEVICE,
    eval=True,
    checkpoint_dir="../../models/checkpoints",
    conf_dir="../../models/dinov3",
)

dv2_model = models["dinov2_s"].eval()
alibi_model = models["alibi_coco_dinov2_s"].eval()
dv2_model.vit.eval()
alibi_model.vit.eval()

# Ensure both DINOv2 and ALiBi-DINOv2 use SDPA (F.scaled_dot_product_attention)
for blk in dv2_model.vit.blocks:
    blk.attn.fused_attn = True
for blk in alibi_model.vit.blocks:
    blk.attn.fused_attn = True

num_heads = dv2_model.vit.blocks[0].attn.num_heads
patch_size = dv2_model.patch_size[0]
n_extra_tokens = 1 + getattr(dv2_model.vit, "num_reg_tokens", 4)
print(
    f"Loaded models (eval={not dv2_model.training and not alibi_model.training}, "
    f"SDPA={dv2_model.vit.blocks[0].attn.fused_attn and alibi_model.vit.blocks[0].attn.fused_attn}) | "
    f"patch_size={patch_size}, num_heads={num_heads}, n_extra_tokens={n_extra_tokens}"
)

In [ ]:
# Multiples of patch_size=14 up to max 1500 px (1498 = 107 * 14)
MAX_IMAGE_SIZE = 1500
image_sizes = [
    s for s in [224, 336, 448, 518, 672, 784, 896, 1036, 1120, 1246, 1344, 1442, 1498] if s <= MAX_IMAGE_SIZE
]


def measure_forward_peak_mb(model, img: Image.Image, device: str = DEVICE) -> float:
    """Measure peak dynamic CUDA memory allocated (in MB, 10^6 bytes) during an eval forward pass,
    returning NaN if the forward pass exceeds available GPU memory (OOM)."""
    import gc

    model.eval()
    model.vit.eval()
    if hasattr(model.vit, "distance_matrix") and model.vit.distance_matrix is not None:
        # Reset cached distance matrix to minimal 1x1 grid so its allocation is captured in peak stats
        model.vit.distance_matrix.update(1, 1, force_update=True)

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize(device)
    torch.cuda.reset_peak_memory_stats(device)
    base_mem = torch.cuda.memory_allocated(device)

    try:
        with torch.inference_mode():
            feat = model.forward_features(img, make_2D=True)
        torch.cuda.synchronize(device)
        peak_mem = torch.cuda.max_memory_allocated(device)
        del feat
        res = (peak_mem - base_mem) / 1e6
    except torch.OutOfMemoryError:
        res = float("nan")
    finally:
        if hasattr(model.vit, "distance_matrix") and model.vit.distance_matrix is not None:
            model.vit.distance_matrix.update(1, 1, force_update=True)
        gc.collect()
        torch.cuda.empty_cache()

    return res


# Warmup CUDA allocator
_warmup_img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
_ = measure_forward_peak_mb(dv2_model, _warmup_img)
_ = measure_forward_peak_mb(alibi_model, _warmup_img)

n_tokens_list: list[int] = []
adj_matrix_mb: list[float] = []
mem_dv2: list[float] = []
mem_alibi: list[float] = []

for s in image_sizes:
    img = Image.fromarray(np.zeros((s, s, 3), dtype=np.uint8))
    n_tok = (s // patch_size) ** 2 + n_extra_tokens
    m_adj = (n_tok**2) * 4 / 1e6  # Single fp32 (N_tokens x N_tokens) matrix in MB

    n_tokens_list.append(n_tok)
    adj_matrix_mb.append(m_adj)

    m_dv2 = measure_forward_peak_mb(dv2_model, img)
    mem_dv2.append(m_dv2)

    m_alibi = measure_forward_peak_mb(alibi_model, img)
    mem_alibi.append(m_alibi)

    diff = m_alibi - m_dv2
    print(
        f"S={s:4d} | N_tok={n_tok:5d} | M_adj={m_adj:6.1f} MB | "
        f"DINOv2 (SDPA)={m_dv2:6.1f} MB | ALiBi-Dv2 (SDPA)={m_alibi:7.1f} MB | "
        f"Delta={diff:7.1f} MB ({diff / m_adj:5.2f} x M_adj)"
    )

### Theoretical Attribution to the $N_{\text{tokens}} \times N_{\text{tokens}}$ Adjacency Matrix (under SDPA)

Let $M_{\text{adj}} = 4\, N_{\text{tokens}}^2$ bytes be the theoretical memory cost of a single fp32 $N_{\text{tokens}} \times N_{\text{tokens}}$ matrix. When both models use `F.scaled_dot_product_attention` (`fused_attn = True`), the memory overhead $\Delta M = M_{\text{ALiBi-Dv2}} - M_{\text{DINOv2}}$ converges to **exactly $1 \times M_{\text{adj}}$**:

1. **DINOv2 (`attn_mask=None`)** uses memory-efficient SDPA with $O(N_{\text{tokens}})$ working memory, never materializing an $N_{\text{tokens}} \times N_{\text{tokens}}$ matrix.
2. **ALiBi-DINOv2 (`attn_mask=bias`)** constructs `distance_matrix.matrix` of shape $(N_{\text{tokens}}, N_{\text{tokens}})$ **in-place** inside a single 8-float-aligned buffer ($\mathbf{1 \times M_{\text{adj}}}$) and passes a zero-copy $(1, 1, N_{\text{tokens}}, N_{\text{tokens}})$ view directly to `F.scaled_dot_product_attention`, which broadcasts across all $H=6$ heads without allocating per-head or alignment-padded copies:

$$\Delta M_{\text{theoretical}} = \mathbf{1 \times M_{\text{adj}}} = 4\, N_{\text{tokens}}^2 \text{ bytes}.$$

In [ ]:
plt.style.use("thesis.mplstyle")
plt.rcParams["text.usetex"] = False
add_custom_font("resources/fonts", "Grotesk")

sizes_arr = np.array(image_sizes)
n_tok_arr = np.array(n_tokens_list)
adj_arr = np.array(adj_matrix_mb)
dv2_arr = np.array(mem_dv2)
alibi_arr = np.array(mem_alibi)
delta_arr = alibi_arr - dv2_arr

# Smooth curve for single adjacency matrix reference
s_smooth = np.linspace(image_sizes[0], image_sizes[-1], 200)
n_tok_smooth = (s_smooth / patch_size) ** 2 + n_extra_tokens
adj_smooth_mb = (n_tok_smooth**2) * 4 / 1e6

COLOR_DV2 = "#785EF0"
COLOR_ALIBI = "#16ce37"
COLOR_THEORY = "#DC267F"
COLOR_ADJ = "#FE6100"

fig, axs = plt.subplots(1, 2, figsize=(7.0, 2.3))

# --- Panel (a): Peak forward memory vs. image side length ---
ax = axs[0]
ax.plot(
    sizes_arr,
    alibi_arr / 1e3,
    "o-",
    color=COLOR_ALIBI,
    label="ALiBi-Dv2 (SDPA, measured)",
    zorder=4,
)
# ax.plot(
#     sizes_arr,
#     (dv2_arr + adj_arr) / 1e3,
#     "--",
#     color=COLOR_THEORY,
#     label=r"DINOv2 + $1 \times M_{\mathrm{adj}}$ (theory)",
#     zorder=5,
# )
ax.plot(
    sizes_arr,
    dv2_arr / 1e3,
    "^-",
    color=COLOR_DV2,
    label="DINOv2 (SDPA, measured)",
    zorder=4,
)
ax.plot(
    s_smooth,
    adj_smooth_mb / 1e3,
    "-.",
    color=COLOR_ADJ,
    label=r"Single $N_{\mathrm{tok}} \times N_{\mathrm{tok}}$ matrix ($M_{\mathrm{adj}}$)",
    zorder=2,
)

ax.set_xlabel(r"Image side length $S$ (px)")
ax.set_ylabel("Peak forward memory (GB)")
ax.set_title("Forward-pass peak GPU memory (SDPA)")
ax.grid(alpha=0.2)
ax.legend(loc="upper left", fontsize=5.5)

# --- Panel (b): Memory overhead normalized by single N_tok x N_tok adjacency matrix ---
ax2 = axs[1]
valid = ~np.isnan(delta_arr)
ax2.plot(
    n_tok_arr[1:][valid[1:]],
    delta_arr[1:][valid[1:]] / adj_arr[1:][valid[1:]],
    "o-",
    color=COLOR_ALIBI,
    label=r"Measured $(M_{\mathrm{ALiBi-Dv2}} - M_{\mathrm{DINOv2}}) / M_{\mathrm{adj}}$",
)

ax2.axhline(
    1.0,
    color=COLOR_THEORY,
    linestyle="--",
    lw=0.9,
    label=r"Theory (single $N_{\mathrm{tok}} \times N_{\mathrm{tok}}$ matrix): $1 \times M_{\mathrm{adj}}$",
)

ax2.set_xlabel(r"Number of tokens $N_{\mathrm{tokens}}$")
ax2.set_ylabel(r"Memory overhead / $M_{\mathrm{adj}}$")
ax2.set_title(r"Attribution to $N_{\mathrm{tokens}} \times N_{\mathrm{tokens}}$ matrix")
ax2.set_ylim(0, 1.5)
ax2.grid(alpha=0.2)
ax2.legend(loc="lower right", fontsize=5.5, ncol=1)

for i, label in enumerate(["(a)", "(b)"]):
    axs[i].text(-0.14, 1.06, label, transform=axs[i].transAxes, fontweight="bold", color="black")

plt.tight_layout()
SAVE = True
if SAVE:
    plt.savefig("saved/R02_memory_cost.pdf", dpi=300, bbox_inches="tight")
    plt.close()
else:
    plt.show()
